# Modular Feature Pipeline Walkthrough (v2)

This notebook demonstrates how to inspect and run each stage of `src.features` independently or through `run_pipeline_from_config` using `configs/pipeline.yaml`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.config import load_pipeline_config
from src.features.validation import validate_and_extract_elements
from src.features.relations import build_relations_and_spatial_summaries
from src.features.posters import build_posters_table
from src.features.schema import generate_feature_schema_metadata
from src.features.pipeline import run_pipeline_from_config

cfg = load_pipeline_config(PROJECT_ROOT / "configs" / "pipeline.yaml")
print(cfg)

## Stage 1: Pool Source Splits, Re-Split 70/15/15, and Validate Annotations

In [ ]:
raw_dir = PROJECT_ROOT / cfg.raw_dir
images_by_split, elements_df, rejected_df, quality_summary = validate_and_extract_elements(
    raw_dir=raw_dir,
    splits=cfg.splits,
    boundary_tolerance_px=cfg.boundary_tolerance_px,
    split_ratios=cfg.split_ratios,
    random_state=cfg.random_state,
    stratify_by_orientation=cfg.stratify_by_orientation,
)
print("Valid elements:", elements_df.shape, "| Rejected:", rejected_df.shape)
print("Split counts:", {k: len(v) for k, v in images_by_split.items()})

## Stage 2 & 3: Build Spatial Relations Graph and Poster Composition Table

In [ ]:
relations_df, poster_rel_stats = build_relations_and_spatial_summaries(
    elements_df,
    align_tol=cfg.align_tol,
    knn_k=cfg.knn_k,
)
posters_df = build_posters_table(
    images_by_split,
    elements_df,
    poster_rel_stats,
    grid_res=cfg.grid_res,
)
schema_meta = generate_feature_schema_metadata(posters_df, elements_df, relations_df)
print("Posters:", posters_df.shape, "| Relations:", relations_df.shape)
print("Numeric poster feature count:", len(schema_meta["tables"]["posters"]["numeric_feature_columns"]))

## Full Orchestrated Execution (Optional Export)

In [ ]:
# Uncomment to run all stages end-to-end and write Parquet/JSON artifacts:
# out = run_pipeline_from_config(cfg)